# Notebook to scrape text of laws

In [ ]:
import os
import time

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from tqdm import tqdm

In [ ]:
target_url = "https://lex.vs.ch/app/fr/texts_of_law/420.1"

to get content of an article

In [ ]:
service = Service()
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=service, options=options)
driver.get(target_url)
time.sleep(2)
html = driver.execute_script("return document.documentElement.outerHTML")
texts = driver.find_elements(By.CLASS_NAME, "document")
document = [element.text for element in texts][0]
# print(driver.page_source)
print(document)
driver.quit()

In [ ]:
with open("document.txt", "w") as f:
    f.write(document)

In [ ]:
def get_text_from_url(url, title):
    service = Service()
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(service=service, options=options)
    driver.get(url)
    time.sleep(3)
    html = driver.execute_script("return document.documentElement.outerHTML")
    texts = driver.find_elements(By.CLASS_NAME, "document")
    document = [element.text for element in texts][0]
    driver.quit()
    with open("documents/" + title.split("-")[0] + ".txt", "w") as f:
        f.write(document)
    return None

to get all urls
you must copy past the tree page from the website in an html file

In [ ]:
path_html = "tree_source.html"
with open(path_html, "r") as f:
    html_content = f.read()
# print(html_content)

In [ ]:
soup = BeautifulSoup(html_content, "html.parser")
print(soup.prettify())

In [ ]:
texts = soup.find_all(class_="d-flex align-items-baseline text-of-law-link")
hrefs = []
titles = []
# Extract text from each element
for text in texts:
    # print(text.get_text(strip=True))
    titles.append(text.get_text(strip=True))
    # print(text.get("href"))
    hrefs.append(text.get("href"))

In [ ]:
for i in range(len(hrefs)):
    href = hrefs[i]
    href = "https://lex.vs.ch" + href
    hrefs[i] = href

### Get all documents

In [ ]:
failed = []
for url, title in tqdm(zip(hrefs, titles), total=len(hrefs)):
    # chek if file already exists
    if os.path.exists("documents/" + title + ".txt") or os.path.exists(
        "documents/" + title.split("-")[0] + ".txt"
    ):
        # print("File already exists")
        continue

    try:
        get_text_from_url(url, title)
    except:
        print("retrying")
        time.sleep(2)
        try:
            get_text_from_url(url, title)
        except:
            print("retrying")
            time.sleep(10)
            try:
                get_text_from_url(url, title)
            except:
                print("failed")
                failed.append(href)
                continue